Import all the necessary modules and load the Data Set.

In [85]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Load the dataset
data = pd.read_csv('parkingLot.csv')

# Display the first few rows of the dataset
# print(data.head())


Basic clean of the original data

In [86]:
# Step 1: Clean the dataset
# Convert 'timestamp' to datetime
data['timestamp'] = pd.to_datetime(data['timestamp'])

# Remove duplicates
data.drop_duplicates(inplace=True)

# Step 2: Filter out entries between 12 AM and 5 AM for both cameras
data = data[(data['timestamp'].dt.hour >= 5) & (data['timestamp'].dt.hour < 24)]

# Step 3: Add a date column for grouping
data['date'] = data['timestamp'].dt.date

Prepare data for part 1

In [87]:
# Filter to only the entry camera
entry_data = data[data['camera_id'] == 1].copy()

# Create a new column for date using .loc to avoid SettingWithCopyWarning
entry_data.loc[:, 'date'] = entry_data['timestamp'].dt.date

# Count total entries per day
daily_entries = entry_data.groupby('date').size().reset_index(name='total_entries')

# Ensure the 'date' column is in datetime format
daily_entries['date'] = pd.to_datetime(daily_entries['date'])

# Sort the date columns to ensure chronological order
daily_entries.sort_values(by='date', inplace=True)

# Create a Sunday indicator (1 for Sundays, 0 for other days)
daily_entries['is_sunday'] = daily_entries['date'].dt.dayofweek == 6  # 6 indicates Sunday in pandas

# Convert the is_sunday column to integers (1 for True, 0 for False)
daily_entries['is_sunday'] = daily_entries['is_sunday'].astype(int)

# Show daily entries
# print("Daily vehicle entries:\n", daily_entries)

Prepare data for second part

In [88]:
# Step 4: Separate entry and exit data
entry_data = data[data['camera_id'] == 1].rename(columns={'date': 'date_entry'})
exit_data = data[data['camera_id'] == 2].rename(columns={'date': 'date_exit'})


# Step 5: Merge entry and exit data on vehicle number and date
merged_data = entry_data.merge(exit_data, 
                                left_on=['vehicle_no', 'date_entry'], 
                                right_on=['vehicle_no', 'date_exit'], 
                                suffixes=('_entry', '_exit'))


# Step 5: Merge entry and exit data on vehicle number and respective dates
merged_data = entry_data.merge(exit_data, 
                                left_on=['vehicle_no', 'date_entry'], 
                                right_on=['vehicle_no', 'date_exit'], 
                                suffixes=('_entry', '_exit'))


# Step 6: Calculate the time spent in the mall
merged_data['time_spent'] = (merged_data['timestamp_exit'] - merged_data['timestamp_entry']).dt.total_seconds() / 60.0  # Convert to minutes

# Step 7: Group by date and calculate the average time spent
average_time_spent = merged_data.groupby('date_entry')['time_spent'].mean().reset_index()

# Ensure the 'date' column is in datetime format for time series analysis
average_time_spent['date_entry'] = pd.to_datetime(average_time_spent['date_entry'])

# print(average_time_spent)

For Part C, implementing outlier smoothing techniques before feeding your data into the model can help improve its performance.\
We are going to use the 2 methods: <br>
1. Z-Score Method <br>
2. IQR Method 

1) Z-Score Method: <br>
This method identifies outliers based on the Z-score, which measures how many standard deviations a data point is from the mean. Outliers can be smoothed by replacing them with the mean or a defined threshold.

In [89]:
def z_score_outlier_smoothing(data, threshold=3):
    """Replace outliers in data with the mean based on Z-scores."""
    # Convert to float to avoid dtype mismatch
    data = data.astype(float)

    # Calculate Z-scores
    z_scores = (data - data.mean()) / data.std()
    
    # Replace outliers with the mean
    smoothed_data = data.copy()
    mean_value = data.mean()
    
    # Replace outliers
    smoothed_data[np.abs(z_scores) > threshold] = mean_value
    
    # Optionally cast back to original data type if desired
    return smoothed_data.astype(data.dtype)  # Cast back to original data type if it makes sense

# Use the function on part 1 data and part 2 data
daily_entries['total_entries_smoothed_z'] = z_score_outlier_smoothing(daily_entries['total_entries'])
average_time_spent['time_spent_smoothed_z'] = z_score_outlier_smoothing(average_time_spent['time_spent'])



2. IQR Method<br>
The Interquartile Range (IQR) method identifies outliers based on the IQR, which is the range between the 25th and 75th percentiles. Outliers are defined as points outside 1.5 times the IQR.

In [90]:
def iqr_outlier_smoothing(data):
    """Replace outliers in data with the mean based on IQR method."""
    # Convert to float to avoid dtype mismatch
    data = data.astype(float)

    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Create a copy to avoid modifying the original data
    smoothed_data = data.copy()

    # Replace outliers with the mean
    mean_value = data.mean()
    smoothed_data[(data < lower_bound) | (data > upper_bound)] = mean_value

    return smoothed_data.astype(data.dtype)  # Cast back to original data type if desired

# Example usage
daily_entries['total_entries_smoothed_iqr'] = iqr_outlier_smoothing(daily_entries['total_entries'])
average_time_spent['time_spent_smoothed_iqr'] = iqr_outlier_smoothing(average_time_spent['time_spent'])


Split it into training and testing data appropriately

In [91]:
# Split the data in part 1 into train and test sets
train_size_1 = len(daily_entries) - 14
train_1, test_1 = daily_entries[:train_size_1], daily_entries[train_size_1:]

# Check the sizes of train and test sets
print(f"\nTrain set size part 1: {len(train_1)}, Test set size part 1: {len(test_1)}")
# print(test_1)

# Split the data in part 1 into train and test sets
train_size_2 = len(average_time_spent) - 7
train_2, test_2 = average_time_spent[:train_size_2], average_time_spent[train_size_2:]

# Check the sizes of train and test sets
print(f"\nTrain set size part 2: {len(train_2)}, Test set size part 2: {len(test_2)}")
# print(test_2)



Train set size part 1: 49, Test set size part 1: 14

Train set size part 2: 56, Test set size part 2: 7


Now lets implement the SARIMA model for PART 1

In [92]:
# Using Z - score

# SARIMAX model with weekly seasonality and Sunday indicator as an exogenous variable
sarima_model_1_z_score = SARIMAX(train_1['total_entries_smoothed_z'], 
                       exog=train_1[['is_sunday']],    # Include Sunday indicator
                       order=(4, 1, 3), 
                       seasonal_order=(1, 1, 1, 6),  # Weekly seasonality
                       enforce_stationarity=False, 
                       enforce_invertibility=False)

# Fit the model
sarima_results_1_z_score = sarima_model_1_z_score.fit(method='powell')

# Forecast the next 14 days using the Sunday indicator as exogenous variable
forecast_values_1_z_score = sarima_results_1_z_score.get_forecast(steps=14, exog=test_1[['is_sunday']])
forecast_1_z_score = forecast_values_1_z_score.predicted_mean

print("\nForecast for total vehicle entries for the next 14 days using Z - score smoothing:")
# print(forecast_1_z_score)

Optimization terminated successfully.
         Current function value: 3.673608
         Iterations: 8
         Function evaluations: 875

Forecast for total vehicle entries for the next 14 days using Z - score smoothing:


In [93]:
# Using IQR Smoothing

# SARIMAX model with weekly seasonality and Sunday indicator as an exogenous variable
sarima_model_1_iqr = SARIMAX(train_1['total_entries_smoothed_iqr'], 
                       exog=train_1[['is_sunday']],    # Include Sunday indicator
                       order=(4, 1, 3), 
                       seasonal_order=(1, 1, 1, 6),  # Weekly seasonality
                       enforce_stationarity=False, 
                       enforce_invertibility=False)

# Fit the model
sarima_results_1_iqr = sarima_model_1_iqr.fit(method='powell')

# Forecast the next 14 days using the Sunday indicator as exogenous variable
forecast_values_1_iqr = sarima_results_1_iqr.get_forecast(steps=14, exog=test_1[['is_sunday']])
forecast_1_iqr = forecast_values_1_iqr.predicted_mean

print("\nForecast for total vehicle entries for the next 14 days using IQR smoothing:")
# print(forecast_1_iqr)


Optimization terminated successfully.
         Current function value: 3.470781
         Iterations: 7
         Function evaluations: 803

Forecast for total vehicle entries for the next 14 days using IQR smoothing:


Now lets implement the SARIMA model for PART 2

In [94]:
# SARIMAX model with seasonal parameters based on your observations using Z-score smoothed data for Part 2
sarima_model_2_z_score = SARIMAX(train_2['time_spent_smoothed_z'], 
                       order=(4, 0, 4),              # Non-seasonal parameters
                       seasonal_order=(1, 0, 1, 10),  # Seasonal parameters with a period of 10 days
                       enforce_stationarity=False, 
                       enforce_invertibility=False)

# Fit the model
sarima_results_2_z_score = sarima_model_2_z_score.fit()

# Forecast the next 14 days
forecast_values_2_z_score = sarima_results_2_z_score.get_forecast(steps=7)  # Forecasting for 14 days
forecast_2_z_score = forecast_values_2_z_score.predicted_mean

print("\nForecast for the average time spent by a person the next 14 days (using Z-score smoothing for Part 2):")
# print(forecast_2_z_score)



Forecast for the average time spent by a person the next 14 days (using Z-score smoothing for Part 2):


In [95]:
# SARIMAX model with seasonal parameters based on your observations using IQR smoothed data for Part 2
sarima_model_2_iqr = SARIMAX(train_2['time_spent_smoothed_iqr'], 
                       order=(4, 0, 4),              # Non-seasonal parameters
                       seasonal_order=(1, 0, 1, 10),  # Seasonal parameters with a period of 10 days
                       enforce_stationarity=False, 
                       enforce_invertibility=False)

# Fit the model
sarima_results_2_iqr = sarima_model_2_iqr.fit()

# Forecast the next 14 days
forecast_values_2_iqr = sarima_results_2_iqr.get_forecast(steps=7)  # Forecasting for 14 days
forecast_2_iqr = forecast_values_2_iqr.predicted_mean

print("\nForecast for the average time spent by a person the next 14 days (using IQR smoothing for Part 2):")
# print(forecast_2_iqr)



Forecast for the average time spent by a person the next 14 days (using IQR smoothing for Part 2):


Now to the calculation part. <br>
Lets write functions for computing MAPE and MASE

In [96]:
# Write a function to perform the calculation: 

def calculate_mape_mase(actual, forecast):
    # Calculate MAE for the model predictions
    mae_model = mean_absolute_error(actual, forecast)

    # Calculate MAPE
    mape = mean_absolute_percentage_error(actual, forecast)

    # Calculate the naive forecast using the last observation method
    naive_forecast = actual.shift(1)  # Shift to simulate naive forecasting
    # Skip the first NaN value in MAE calculation
    mae_naive = mean_absolute_error(actual[1:], naive_forecast[1:])  

    # Calculate MASE
    mase = mae_model / mae_naive
    
    return mape, mase

# Part 1
mape_1_z_score, mase_1_z_score = calculate_mape_mase(test_1['total_entries'], forecast_1_z_score)
mape_1_iqr, mase_1_iqr = calculate_mape_mase(test_1['total_entries'], forecast_1_iqr)

# Part 2
mape_2_z_score, mase_2_z_score = calculate_mape_mase(test_2['time_spent'], forecast_2_z_score)
mape_2_iqr, mase_2_iqr = calculate_mape_mase(test_2['time_spent'], forecast_2_iqr)

# Print results
print(f"Part 1 - Z-score: MAPE: {mape_1_z_score:.4f}, MASE: {mase_1_z_score:.4f}")
print(f"Part 1 - IQR: MAPE: {mape_1_iqr:.4f}, MASE: {mase_1_iqr:.4f}")
print(f"Part 2 - Z-score: MAPE: {mape_2_z_score:.4f}, MASE: {mase_2_z_score:.4f}")
print(f"Part 2 - IQR: MAPE: {mape_2_iqr:.4f}, MASE: {mase_2_iqr:.4f}")


Part 1 - Z-score: MAPE: 0.0500, MASE: 0.7442
Part 1 - IQR: MAPE: 0.0533, MASE: 0.7911
Part 2 - Z-score: MAPE: 0.0062, MASE: 0.0921
Part 2 - IQR: MAPE: 0.0062, MASE: 0.0921
